# 16. 당뇨병 예측 모델 해석하기

이번 장에서는 모델이 어떤 입력 특성에 민감하게 반응하는지 확인합니다.

중요한 주의:

```text
모델 해석은 인과관계 증명이 아닙니다.
특성이 중요하게 나왔다고 해서 그 특성이 질병의 원인이라는 뜻은 아닙니다.
```

## 1. 라이브러리 불러오기

이번 장에서는 모델 학습, 평가, 특성 중요도 계산을 함께 다룹니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout
from keras.callbacks import EarlyStopping

## 2. 데이터 준비

13장부터 사용한 당뇨병 이진 분류 데이터를 다시 준비합니다.

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\diabetes_or_cardiovascular\diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
TARGET = "Diabetes_binary"

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET])
y = df[TARGET]

feature_names = list(X.columns)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("특성 개수:", len(feature_names))
print("특성 이름:")
print(feature_names)

## 3. 해석용 MLP 모델 만들기

이번 장의 목적은 최고 성능이 아니라 해석 흐름입니다.

그래도 너무 약한 모델이면 해석이 의미 없어질 수 있으므로, 이전 장과 비슷한 MLP를 사용합니다.

In [ ]:
input_dim = X_train_scaled.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

model.summary()

## 4. 모델 학습

이 장에서는 학습 결과를 바탕으로 특성 중요도를 계산합니다.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

## 5. 기준 성능 계산

Permutation Importance를 계산하려면 먼저 원래 검증 성능을 알아야 합니다.

여기서는 F1 점수를 기준 성능으로 사용합니다.

In [ ]:
def predict_label(model, X_scaled, threshold=0.5):
    """모델 예측 확률을 0/1 라벨로 바꿉니다."""
    y_prob = model.predict(X_scaled, verbose=0).ravel()
    return (y_prob >= threshold).astype(int)


y_val_pred = predict_label(model, X_val_scaled, threshold=0.5)

base_accuracy = accuracy_score(y_val, y_val_pred)
base_precision = precision_score(y_val, y_val_pred, zero_division=0)
base_recall = recall_score(y_val, y_val_pred, zero_division=0)
base_f1 = f1_score(y_val, y_val_pred, zero_division=0)

print(f"기준 accuracy:  {base_accuracy:.4f}")
print(f"기준 precision: {base_precision:.4f}")
print(f"기준 recall:    {base_recall:.4f}")
print(f"기준 f1:        {base_f1:.4f}")

## 6. Permutation Importance 계산 함수

특정 컬럼 하나를 섞고 성능이 얼마나 떨어지는지 계산합니다.

성능이 많이 떨어지면 모델이 그 컬럼에 많이 의존했을 가능성이 있습니다.

In [ ]:
def permutation_importance_for_feature(model, X_val_original, y_val, feature_name, scaler, base_score, random_state=42):
    """특정 특성 하나를 섞었을 때 F1 점수가 얼마나 떨어지는지 계산합니다."""
    
    # 원본 검증 데이터를 복사합니다.
    # copy()를 쓰는 이유는 원본 X_val을 망가뜨리지 않기 위해서입니다.
    X_permuted = X_val_original.copy()
    
    # 난수 생성기를 만듭니다.
    rng = np.random.default_rng(random_state)
    
    # 선택한 컬럼의 값만 섞습니다.
    # 다른 컬럼은 그대로 두고 이 컬럼만 관계를 깨뜨립니다.
    X_permuted[feature_name] = rng.permutation(X_permuted[feature_name].values)
    
    # 섞은 데이터를 기존 scaler 기준으로 변환합니다.
    X_permuted_scaled = scaler.transform(X_permuted)
    
    # 섞인 데이터로 예측하고 F1을 계산합니다.
    y_pred = predict_label(model, X_permuted_scaled, threshold=0.5)
    permuted_score = f1_score(y_val, y_pred, zero_division=0)
    
    # 원래 점수에서 섞은 뒤 점수를 뺍니다.
    importance = base_score - permuted_score
    
    return importance, permuted_score

## 7. 모든 특성의 중요도 계산

각 컬럼을 하나씩 섞어 보며 F1 점수가 얼마나 떨어지는지 기록합니다.

In [ ]:
importance_rows = []

for feature_name in feature_names:
    importance, permuted_score = permutation_importance_for_feature(
        model=model,
        X_val_original=X_val,
        y_val=y_val,
        feature_name=feature_name,
        scaler=scaler,
        base_score=base_f1,
        random_state=42
    )
    
    importance_rows.append({
        "feature": feature_name,
        "base_f1": base_f1,
        "permuted_f1": permuted_score,
        "importance": importance,
    })

importance_df = pd.DataFrame(importance_rows)

# importance가 큰 특성이 위로 오게 정렬합니다.
importance_df = importance_df.sort_values("importance", ascending=False)

importance_df.head(10)

In [ ]:
top_n = 10
top_importance = importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(8, 5))
plt.barh(top_importance["feature"], top_importance["importance"])
plt.title("Top Permutation Importance")
plt.xlabel("F1 decrease after permutation")
plt.tight_layout()
plt.show()

## 8. 중요도 해석하기

중요도가 높다는 말은 다음 뜻에 가깝습니다.

```text
이 컬럼을 섞었더니 모델 성능이 많이 떨어졌다.
따라서 이 모델은 이 컬럼을 예측에 많이 사용했을 가능성이 있다.
```

하지만 이것은 원인이라는 뜻이 아닙니다.

In [ ]:
print("상위 중요 특성:")

for _, row in importance_df.head(5).iterrows():
    print(f"- {row['feature']}: F1 하락 {row['importance']:.4f}")

print("\n주의:")
print("이 결과는 인과관계가 아니라 모델의 예측 의존도를 관찰한 것입니다.")

## 9. 한 사람의 입력값 바꿔보기

이제 특정 샘플 하나를 골라 입력값을 바꾸면 예측 점수가 어떻게 달라지는지 봅니다.

예시로 BMI 값을 바꿔보겠습니다.

In [ ]:
# 검증 데이터에서 첫 번째 사람을 하나 선택합니다.
sample = X_val.iloc[0].copy()

print("원래 샘플:")
display(sample.to_frame(name="value"))

In [ ]:
def predict_one_sample_probability(model, sample_series, scaler):
    """한 사람의 입력값을 받아 당뇨병 있음 예측 점수를 반환합니다."""
    
    # Series를 DataFrame 한 행으로 바꿉니다.
    sample_df = pd.DataFrame([sample_series])
    
    # 학습 데이터 기준으로 만든 scaler를 사용해 변환합니다.
    sample_scaled = scaler.transform(sample_df)
    
    # predict()는 2차원 배열 형태의 예측값을 반환하므로 [0][0]으로 숫자 하나를 꺼냅니다.
    probability = model.predict(sample_scaled, verbose=0)[0][0]
    
    return float(probability)


original_probability = predict_one_sample_probability(model, sample, scaler)
print(f"원래 예측 점수: {original_probability:.4f}")

## 10. BMI 변화에 따른 예측 점수

BMI 값을 여러 값으로 바꿔보며 모델 예측 점수가 어떻게 움직이는지 확인합니다.

이것은 모델 반응을 보는 실험입니다.

In [ ]:
bmi_values = [20, 25, 30, 35, 40]
bmi_result_rows = []

for bmi in bmi_values:
    changed_sample = sample.copy()
    
    # BMI 값만 바꿉니다.
    changed_sample["BMI"] = bmi
    
    probability = predict_one_sample_probability(model, changed_sample, scaler)
    
    bmi_result_rows.append({
        "BMI": bmi,
        "predicted_probability": probability,
    })

bmi_result_df = pd.DataFrame(bmi_result_rows)
bmi_result_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(bmi_result_df["BMI"], bmi_result_df["predicted_probability"], marker="o")
plt.title("Prediction Change by BMI")
plt.xlabel("BMI")
plt.ylabel("Predicted probability for diabetes")
plt.ylim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.show()

## 11. 다른 특성도 바꿔보기

이번에는 `HighBP` 값을 0과 1로 바꿔봅니다.

`HighBP`는 고혈압 여부를 나타내는 0/1 특성입니다.

In [ ]:
highbp_rows = []

for highbp in [0, 1]:
    changed_sample = sample.copy()
    changed_sample["HighBP"] = highbp
    
    probability = predict_one_sample_probability(model, changed_sample, scaler)
    
    highbp_rows.append({
        "HighBP": highbp,
        "predicted_probability": probability,
    })

highbp_result_df = pd.DataFrame(highbp_rows)
highbp_result_df

## 12. 해석 문장 만들기

해석 결과는 조심스럽게 표현해야 합니다.

아래 템플릿은 포트폴리오에 사용할 수 있는 안전한 문장입니다.

In [ ]:
print("해석 템플릿")
print("- permutation importance를 사용해 각 특성을 섞었을 때 F1 점수가 얼마나 하락하는지 확인했다.")
print("- 중요도가 높게 나온 특성은 이 모델이 예측에 많이 의존했을 가능성이 있는 특성으로 해석했다.")
print("- 특정 샘플에서 BMI 값을 바꾸며 모델 예측 점수의 변화를 관찰했다.")
print("- 단, 이 결과는 인과관계가 아니라 모델 반응과 예측 의존도에 대한 해석이다.")

## 13. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. 모델 해석은 성능 점수만 보는 것보다 프로젝트를 더 깊게 만든다.
2. Permutation Importance는 컬럼을 섞어 성능 하락을 보는 방법이다.
3. 중요도가 높다는 말은 원인이라는 뜻이 아니다.
4. 한 사람의 입력값을 바꿔 예측 점수 변화를 관찰할 수 있다.
5. 건강 데이터에서는 해석 표현을 조심해야 한다.
```

## 과제

1. importance_df에서 상위 5개 특성을 확인하고, 왜 중요하게 나왔을지 추측해보세요.
2. BMI 말고 다른 숫자 특성 하나를 골라 값을 바꿔보세요.
3. 0/1 특성 하나를 골라 값을 바꿔보세요.
4. “인과가 아니라 모델 반응”이라는 표현이 왜 중요한지 설명해보세요.